Getting started with kernel profiling
=====================================

This notebook shows how to get started with kernel profiling.

Kernel profiling is the process of collecting detailed performance metrics to understand how kernels utilize GPU hardware and how closely they approach peak performance.
It helps identify performance bottlenecks and assess the impact of optimizations.
NVIDIA provides [Nsight Compute](https://developer.nvidia.com/nsight-compute) for this purpose.
`ReProspect` enables a fully programmatic use of this tool: it launches it, reads the collected performance metrics into Python data structures, and supports analyses of these data that can go all the way to test assertions.

As an example, we consider a kernel that fills a buffer.
In the reference case, each thread writes a single element.
We programmatically analyze how restructuring the kernel so that each thread writes a batch of several elements, the *static batch size*, a compile-time parameter of the kernel, can improve performance.

About this page
---------------

This page is rendered from a Jupyter notebook, executed at documentation build time, located at `docs/source/getting-started/example_kernel_profiling.ipynb` in the repository.

To run the example yourself, {download}`download the notebook <example_kernel_profiling.ipynb>` and open it in JupyterLab.
Alternatively, copy-paste the code snippets successively into an interactive Python session.

The requirements are Python 3.10 or newer and `ReProspect`.

The example also requires:
- a CUDA Toolkit installation providing `nvcc`;
- an [Nsight Compute installation](https://developer.nvidia.com/tools-overview/nsight-compute/get-started) providing the command-line tool `ncu`;
- the NVTX header `nvtx3/nvtx3.hpp` (included in recent CUDA Toolkits; otherwise available from [NVTX installation](https://github.com/NVIDIA/NVTX#how-do-i-get-nvtx));
- a C++20-capable toolchain.

At the end, a final section "Going further" additionally uses the binary-analysis component and requires that the CUDA Toolkit installation provide `cuobjdump` and `cu++filt`.
That final section is optional and not needed for the rest of the notebook.

Because kernel profiling executes the program, a GPU is required.
In addition, kernel profiling requires sufficiently elevated access privileges (e.g., `--cap-add=SYS_ADMIN` for a Docker container; see also [NVIDIA's documentation](https://developer.nvidia.com/nvidia-development-tools-solutions-err_nvgpuctrperm-permission-issue-performance-counters)).

Source code
-----------

We consider a kernel that writes a given value to each element of a buffer.
The kernel is parameterized by the compile-time parameter `StaticBatchSize`.
In the reference case, which corresponds to a static batch size equal to 1, each thread writes a single element:

```{tikz} Reference kernel (StaticBatchSize equal to 1).
:align: center

\begin{tikzpicture}[
    x=0.95cm, y=0.95cm,
    warpcell/.style={draw, minimum width=0.475cm, minimum height=0.475cm,
                 inner sep=0pt, fill=gray!6, line width=0.8pt},
    buffercell/.style={draw, minimum width=0.475cm, minimum height=0.95cm,
                 inner sep=0pt, fill=gray!6, line width=0.8pt},
    targetcell/.style={buffercell, fill=blue!15},
    dotslabel/.style={font=\Large},
    threadlabel/.style={font=\small, inner sep=1pt},
    arrow/.style={->, line width=0.8pt, shorten >=1.5pt, shorten <=1.5pt}
]
    \node[] (W) at (-0.5,0) {Warp:};
    \foreach \i in {0,...,5} {\node[warpcell] (W\i) at (1+\i*0.5,0) {};}
    \node[dotslabel] at (4.5,0) {$\cdots$};
    \foreach \i in {9,10} {\node[warpcell] (W\i) at (1+\i*0.5,0) {};}
    \node[threadlabel] (l0) at ([yshift=6] W0.north) {0};
    \node[threadlabel] (l1) at ([yshift=6] W1.north) {1};
    \node[threadlabel] (l10) at ([yshift=6] W10.north) {31};

    \node[] (B) at (-0.5,-3) {Buffer:};
    \foreach \i in {0,...,5} {\node[targetcell] (B\i) at (1+\i*0.5,-3) {};}
    \node[dotslabel] at (4.5,-3) {$\cdots$};
    \foreach \i in {9,10} {\node[targetcell] (B\i) at (1+\i*0.5,-3) {};}
    \node[dotslabel] at (7.5,-3) {$\cdots$};
    \foreach \i in {16,17} {\node[buffercell] (B\i) at (1+\i*0.5,-3) {};}
    \foreach \i in {18,...,23} {\node[buffercell] (B\i) at (1+\i*0.5,-3) {};}
    \node[dotslabel] at (13.5,-3) {$\cdots$};
    \foreach \i in {27,28} {\node[buffercell] (B\i) at (1+\i*0.5,-3) {};}
    \node[dotslabel] at (16.5,-3) {$\cdots$};
    \foreach \i in {34,35} {\node[buffercell] (B\i) at (1+\i*0.5,-3) {};}

    \foreach \i in {0,...,5} {\draw[arrow] (W\i.south) -- (B\i.north);}
    \foreach \i in {9,10} {\draw[arrow] (W\i.south) -- (B\i.north);}

    \draw[dashed, line width=0.7pt] (B0.south west)  -- ++(0,-1.2);
    \draw[dashed, line width=0.7pt] (B35.south east) -- ++(0,-1.2);

    \draw[<->, line width=0.8pt] ([yshift=-1.0cm]B0.south west) -- ([yshift=-1.0cm]B35.south east)
        node[midway, below=2pt, font=\small]
        {\texttt{size}};
\end{tikzpicture}
```

For values of the static batch size larger than 1, the kernel is restructured so that each thread performs `StaticBatchSize` writes to elements separated by `work_stride`, the total number of threads launched, as illustrated here for the case of a static batch size of 2:

```{tikz} Restructured kernel (StaticBatchSize equal to 2).
:align: center

\begin{tikzpicture}[
    x=0.95cm, y=0.95cm,
    warpcell/.style={draw, minimum width=0.475cm, minimum height=0.475cm,
                 inner sep=0pt, fill=gray!6, line width=0.8pt},
    buffercell/.style={draw, minimum width=0.475cm, minimum height=0.95cm,
                 inner sep=0pt, fill=gray!6, line width=0.8pt},
    targetcell/.style={buffercell, fill=blue!15},
    dotslabel/.style={font=\Large},
    threadlabel/.style={font=\small, inner sep=1pt},
    arrow/.style={->, line width=0.8pt, shorten >=1.5pt, shorten <=1.5pt}
]
    \node[] (W) at (-0.5,0) {Warp:};
    \foreach \i in {0,...,5} {\node[warpcell] (W\i) at (1+\i*0.5,0) {};}
    \node[dotslabel] at (4.5,0) {$\cdots$};
    \foreach \i in {9,10} {\node[warpcell] (W\i) at (1+\i*0.5,0) {};}
    \node[threadlabel] (l0) at ([yshift=6] W0.north) {0};
    \node[threadlabel] (l1) at ([yshift=6] W1.north) {1};
    \node[threadlabel] (l10) at ([yshift=6] W10.north) {31};

    \node[] (B) at (-0.5,-3) {Buffer:};
    \foreach \i in {0,...,5} {\node[targetcell] (B\i) at (1+\i*0.5,-3) {};}
    \node[dotslabel] at (4.5,-3) {$\cdots$};
    \foreach \i in {9,10} {\node[targetcell] (B\i) at (1+\i*0.5,-3) {};}
    \node[dotslabel] at (7.5,-3) {$\cdots$};
    \foreach \i in {16,17} {\node[buffercell] (B\i) at (1+\i*0.5,-3) {};}
    \foreach \i in {18,...,23} {\node[targetcell] (B\i) at (1+\i*0.5,-3) {};}
    \node[dotslabel] at (13.5,-3) {$\cdots$};
    \foreach \i in {27,28} {\node[targetcell] (B\i) at (1+\i*0.5,-3) {};}
    \node[dotslabel] at (16.5,-3) {$\cdots$};
    \foreach \i in {34,35} {\node[buffercell] (B\i) at (1+\i*0.5,-3) {};}

    \foreach \i in {0,...,5} {\draw[arrow] (W\i.south) -- (B\i.north);}
    \foreach \i in {9,10} {\draw[arrow] (W\i.south) -- (B\i.north);}

    \foreach \i/\j in {0/18,1/19,2/20,3/21,4/22,5/23,9/27,10/28} {\draw[arrow] (W\i.south) -- (B\j.north);}

    \draw[dashed, line width=0.7pt] (B0.south west)  -- ++(0,-2.4);
    \draw[dashed, line width=0.7pt] (B18.south west) -- ++(0,-1.2);
    \draw[dashed, line width=0.7pt] (B35.south east) -- ++(0,-2.4);

    \draw[<->, line width=0.8pt] ([yshift=-1.0cm]B0.south west) -- ([yshift=-1.0cm]B18.south west)
        node[midway, below=2pt, font=\small] {\texttt{work\_stride}};
    \draw[<->, line width=0.8pt] ([yshift=-1.0cm]B18.south west) -- ([yshift=-1.0cm]B35.south east)
        node[midway, below=2pt, font=\small] {\texttt{work\_stride}};
    \draw[<->, line width=0.8pt] ([yshift=-2.1cm]B0.south west) -- ([yshift=-2.1cm]B35.south east)
        node[midway, below=2pt, font=\small]
        {\texttt{size} $=$ \texttt{work\_stride} $\times$ 2};
\end{tikzpicture}
```

The program launches the kernel to fill a buffer of 512 Mi elements ({math}`1024 \times 1024 \times 512`), for elements of type `char` and of type `int`, corresponding to buffers of 0.5 GiB and 2 GiB, respectively, and for static batch sizes of 1, 2, 4, 8, 16 and 32.

The source code features NVTX annotations, for two purposes.
On the one hand, the outer NVTX range `start_end_range` delimits the kernel launches that Nsight Compute profiles: only launches within this range are collected.
In particular, the warm-up launches fall outside `start_end_range` and are not profiled.
On the other hand, the inner ranges provide convenient accessor paths for the collected data.
In particular, with the inner ranges `char_range` and `int_range`, each in turn with nested ranges `static_batch_size_1`, ..., `static_batch_size_32`, each kernel launch falls within a specific nested range.
Below, we will look up the collected performance metrics by these ranges.
The ranges belong to a dedicated NVTX domain, which keeps them separate from other NVTX annotations, such as annotations that libraries used by an application may emit themselves.

The program also times each kernel execution and computes the corresponding throughput (GB/s).
Upon passing the option `--print-throughputs`, it prints these throughputs.

In [ ]:
CODE = """
#include <array>
#include <cassert>
#include <chrono>
#include <format>
#include <iostream>
#include <source_location>
#include <sstream>
#include <stdexcept>
#include <string>
#include <string_view>
#include <utility>

#include <cuda_runtime.h>
#include <nvtx3/nvtx3.hpp>

inline void check_cudart_call(
    const cudaError_t status,
    const char* const statement,
    const std::source_location& loc = std::source_location::current()) {
    if (status != cudaSuccess) {
        std::ostringstream oss;
        oss << statement << " failed: " << status << " (" << cudaGetErrorName(status)
            << "): " << cudaGetErrorString(status) << " (" << loc.file_name() << ":" << loc.line() << ")";

        throw std::runtime_error(oss.str());
    }
}

#define CHECK_CUDART_CALL(statement) check_cudart_call((statement), #statement)

struct ExampleKernelProfilingDomain {
    static constexpr char const * name{"example_kernel_profiling_domain"};
};

template <unsigned int StaticBatchSize, typename T>
__global__ void fill_kernel(T* data, const T val, const unsigned int size) {
    const auto work_stride = blockDim.x * gridDim.x;
    const auto batch_stride = work_stride * StaticBatchSize;

    const unsigned int iwork = threadIdx.x + blockDim.x * blockIdx.x;
    for (unsigned int i = 0; i < batch_stride && i < size - iwork; i += work_stride) {
        data[iwork + i] = val;
    }
}

template <unsigned int StaticBatchSize, typename T>
void run_fill(cudaStream_t stream, T* data, const T val, const unsigned int size) {
    const std::string name = std::format("static_batch_size_{}", StaticBatchSize);
    const nvtx3::scoped_range_in<ExampleKernelProfilingDomain> range(name);

    constexpr unsigned int block_size = 128;

    assert(size % StaticBatchSize == 0);
    const unsigned int nwork = size / StaticBatchSize;

    const dim3 block(block_size, 1, 1);
    const dim3 grid((nwork + block_size - 1) / block_size, 1, 1);

    fill_kernel<StaticBatchSize><<<grid, block, 0, stream>>>(data, val, size);
    CHECK_CUDART_CALL(cudaStreamSynchronize(stream));
}

template <typename T, unsigned int... StaticBatchSizes>
auto timed_sweep(std::integer_sequence<unsigned int, StaticBatchSizes...>,
                 cudaStream_t stream, std::string_view label, const T val, const unsigned int size) {
    const std::string name = std::format("{}_range", label);
    const nvtx3::scoped_range_in<ExampleKernelProfilingDomain> range(name);

    T* data;
    CHECK_CUDART_CALL(cudaMallocAsync(&data, size * sizeof(T), stream));
    CHECK_CUDART_CALL(cudaStreamSynchronize(stream));

    const auto timed = [](auto&& f) {
        const auto start = std::chrono::high_resolution_clock::now();
        f();
        const auto end = std::chrono::high_resolution_clock::now();
        return end - start;
    };

    std::array<std::chrono::duration<double>, sizeof...(StaticBatchSizes)> timings{
        timed([&] { run_fill<StaticBatchSizes, T>(stream, data, val, size); })...
    };

    CHECK_CUDART_CALL(cudaFreeAsync(data, stream));
    CHECK_CUDART_CALL(cudaStreamSynchronize(stream));

    return timings;
}

int main(int argc, char* argv[]) {
    using StaticBatchSizeSequence = std::integer_sequence<unsigned int, 1, 2, 4, 8, 16, 32>;

    constexpr unsigned int size = 1024 * 1024 * 512;

    cudaStream_t stream;
    CHECK_CUDART_CALL(cudaStreamCreate(&stream));

    // Warmup. Avoid measuring module loading and other overhead.
    timed_sweep<char>(StaticBatchSizeSequence{}, stream, "char", 1, size);
    timed_sweep<int >(StaticBatchSizeSequence{}, stream, "int",  1, size);

    const auto start_end_range = nvtx3::start_range_in<ExampleKernelProfilingDomain>("start_end_range");

    const auto timings_char = timed_sweep<char>(StaticBatchSizeSequence{}, stream, "char", 1, size);
    const auto timings_int  = timed_sweep<int >(StaticBatchSizeSequence{}, stream, "int",  1, size);

    nvtx3::end_range_in<ExampleKernelProfilingDomain>(start_end_range);

    if (argc == 2 && std::string(argv[1]) == "--print-throughputs") {
        // Header.
        std::cout << std::format("{:<24}", "Static batch size");
        [&]<unsigned int... StaticBatchSizes>(std::integer_sequence<unsigned int, StaticBatchSizes...>) {
             ((std::cout << std::format(" | {:>7}", StaticBatchSizes)), ...);
        }(StaticBatchSizeSequence{});
        std::cout << std::endl;

        // Row char.
        std::cout << std::format("{:<24}", "Throughput - char (GB/s)");
        for (const auto& timing : timings_char) {
            std::cout << std::format(" | {:>7.1f}", sizeof(char) * size / (timing.count() * 1e9));
        }
        std::cout << std::endl;

        // Row int.
        std::cout << std::format("{:<24}", "Throughput - int (GB/s)");
        for (const auto& timing : timings_int) {
            std::cout << std::format(" | {:>7.1f}", sizeof(int) * size / (timing.count() * 1e9));
        }
        std::cout << std::endl;
    }

    CHECK_CUDART_CALL(cudaStreamDestroy(stream));
}
"""

Compilation
-----------

Because this example will execute the program, we compile the source code for the native architecture, i.e., the architecture of the GPU present on the machine.

In [ ]:
import pathlib
import subprocess

from reprospect.utils import rich_helpers
from reprospect.utils.detect import GPUDetector

print(subprocess.check_output(('nvcc', '--version')).decode().strip())

print(subprocess.check_output(('ncu', '--version')).decode().strip())

visible_gpus = GPUDetector().detect()
print(f'Visible GPUs:\n{rich_helpers.to_string(rich_helpers.df_to_table(visible_gpus))}')

workdir = pathlib.Path.cwd() / 'example_kernel_profiling'
workdir.mkdir(exist_ok=True)

source = workdir / 'static_batch_size.cu'
executable = workdir / 'static_batch_size'

source.write_text(CODE)
_ = subprocess.check_call(('nvcc', '-arch=native', '-std=c++20', '-O3', '-o', executable, source))

Measured throughputs
--------------------

Let us run the program with the option `--print-throughputs`.
This is a bare, unprofiled run: execution times measured by the program itself while profiling are not meaningful because the profiler may replay kernels and lock clocks, as described [here](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html?highlight=clock#reproducibility).
Our program performs a warmup phase that launches each kernel once before the timed sweep, so that one-time effects such as lazy module loading do not affect the time measurements.

In [ ]:
_ = subprocess.check_call((executable, '--print-throughputs'))

We can observe that for the type `char`, the reference kernel (with a static batch size of 1) achieves a throughput that is about three times lower than the highest throughputs measured across the scenarios.
The measured throughput increases with the batch size and plateaus from a batch size of 4.
For the type `int`, the reference kernel already achieves the plateau performance.
The plateau is common to both element types.

It is tempting to hypothesize that the batching improves the memory access pattern.
However, as the figures above illustrate, the reference kernel is already such that consecutive lanes of a warp write to consecutive elements in the buffer, and the restructured kernels preserve this memory access pattern.
Improved coalescing thus cannot explain the increased throughput.

The kernel profiling that follows will let us assert that the memory accesses are coalesced at every static batch size, and identify the mechanism that actually drives the performance improvement.

Kernel profiling metrics
------------------------

Nsight Compute [categorizes](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-structure) metrics into counter metrics, ratio metrics and throughput metrics.
Counter metrics have four sub-metrics under them: the so-called roll-ups `.sum`, `.avg`, `.min` and `.max`, which represent different ways of aggregating counts across all instances of the hardware unit that the metric is associated with.
For instance, the counter metric `smsp__inst_executed.sum` represents the sum of all warp-level instructions executed across all [streaming multiprocessor sub-partitions (SMSPs)](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#streaming-multiprocessor).
Sub-metrics can in turn have further sub-metrics under them, holding derived quantities calculated by Nsight Compute.
For instance, Nsight Compute has a database with peak values that certain sub-metrics may reach, and it can thus calculate for such sub-metrics the percentage of the peak that the collected value attains (e.g. `.pct_of_peak_sustained_elapsed`).
Ratio metrics have the roll-ups `.pct`, `.ratio` and `.max_rate` instead.
Throughput metrics always require sub-metric paths with multiple components.

`ReProspect` represents such metrics as typed objects.
They provide the *names* that must be passed to Nsight Compute to request collection, as well as the *labels* under which `ReProspect` will store the collected values for analysis.
A counter metric is constructed from its Nsight Compute base name, an optional human-readable pretty name, and one or several typed sub-metric paths ({py:class}`reprospect.tools.ncu.metrics.MetricCounter`).
The Nsight Compute names are assembled by joining the base name with each sub-metric path, and the labels are obtained by joining the pretty name with a pretty rendering of the sub-metric paths, omitting sub-metric path components that `ReProspect` considers default, namely, `.sum` for counter metrics and `.ratio` for ratio metrics.

In [ ]:
from reprospect.tools.ncu import MetricCounter, MetricCounterRollUp

inst_executed = MetricCounter(name='smsp__inst_executed', pretty_name='Executed instructions', subs=(MetricCounterRollUp.SUM,))

inst_executed_labels, inst_executed_names = inst_executed.labels(), inst_executed.gather()
print(inst_executed_labels, inst_executed_names)

For commonly used metrics, `ReProspect` provides targeted factories that encapsulate the Nsight Compute names and human-readable pretty names, such as the class {py:class}`reprospect.tools.ncu.metrics.L1TEXCache` for metrics for memory workload analysis.

The metrics for this example combine factory-provided and directly constructed metrics:

In [ ]:
from reprospect.tools.ncu import (
    L1TEXCache,
    LaunchGrid,
    MetricCounterRollUpQuantity,
    WarpStall,
    gather,
    labels,
)

metrics = (
    # Launch grid sizes.
    *LaunchGrid.create(dims=('x',)),
    # Overall instruction count.
    inst_executed,
    # L1/TEX cache memory traffic.
    *L1TEXCache.GlobalStore.Instructions.create(),
    *L1TEXCache.GlobalStore.Requests.create(),
    *L1TEXCache.GlobalStore.Sectors.create(),
    # Warp stall reasons.
    *WarpStall.ShortScoreboard.create(),
    *WarpStall.LGThrottle.create(),
    *WarpStall.LongScoreboard.create(),
    # DRAM memory traffic.
    MetricCounter(name='dram__bytes_op_write', pretty_name='Device memory store utilization', subs=((MetricCounterRollUp.SUM, MetricCounterRollUpQuantity.PCT_OF_PEAK_SUSTAINED_ELAPSED),)),
)

metric_labels, metric_names = labels(metrics), gather(metrics)

print(f'Metrics:\n{rich_helpers.to_string(rich_helpers.rows_to_table(zip(metric_labels, metric_names, strict=True), columns=("Label", "Name")))}')

The requested L1/TEX cache memory traffic metrics are counter metrics that provide respectively the number of store instructions the SMSPs execute to write to global memory, the number of requests these store instructions generate to the memory system, and the number of sectors, i.e., aligned contiguous 32-byte chunks of global memory, these requests access; see also Nsight Compute's documentation on the [hardware model](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html?highlight=average#hardware-model), as well as this [presentation](https://www.nvidia.com/en-us/on-demand/session/gtcspring21-s32089/).

The requested warp stall reason metrics are ratio metrics that provide the average number of cycles that warps spend in the associated stalled state per issued instruction.

Running kernel profiling
------------------------

Nsight Compute provides the [command-line tool](https://docs.nvidia.com/nsight-compute/NsightComputeCli/index.html) `ncu` for collecting performance metrics. 
Here, we invoke it through the `ReProspect` class {py:class}`reprospect.tools.ncu.session.Session`.
The argument `executable` designates the executable on which to collect data.
The argument `nvtx_includes` specifies one or several NVTX ranges to delimit the kernel launches that `ncu` profiles.
The value `'example_kernel_profiling_domain@start_end_range'` follows the `ncu` [convention](https://docs.nvidia.com/nsight-compute/NsightComputeCli/index.html#nvtx-filtering) `<domain>@<range>`.
The argument `output` determines the output file; for the passed value `workdir / executable.name`, the report is written to `workdir / f'{executable.name}.ncu-rep'`.

In [ ]:
from reprospect.tools.ncu import Command, Session

nc = Session(
    command=Command(
        executable=executable,
        output=workdir / executable.name,
        metrics=metrics,
        nvtx_includes=('example_kernel_profiling_domain@start_end_range',),
    ),
)

nc.run(cwd=workdir)

Kernel profiling results
------------------------

Nsight Compute [provides](https://docs.nvidia.com/nsight-compute/PythonReportInterface/index.html) the Python module `ncu_report` for low-level access to the output file generated by `ncu`.
The `ReProspect` class {py:class}`reprospect.tools.ncu.report.Report` relies on `ncu_report` and adds infrastructure around it. 
Its method {py:meth}`~reprospect.tools.ncu.report.Report.extract_results_in_range` retrieves the collected performance metrics.

In [ ]:
from reprospect.tools.ncu import Report

report = Report(command=nc.command)

results = report.extract_results_in_range(metrics=metrics)

The method {py:meth}`~reprospect.tools.ncu.report.Report.extract_results_in_range` returns a {py:class}`~reprospect.tools.ncu.report.ProfilingResults` hierarchical data structure, in which the collected performance metrics are organised by nested NVTX range.
Each leaf node holds the performance metrics collected for one kernel launch, as a mapping from metric label to value.

The profiling results can then be queried by their NVTX path:

In [ ]:
results_char = results.query(accessors=('char_range',))

print(results_char)

Between an innermost NVTX range and the metrics for a kernel launch, the hierarchy contains one more intermediate level.
In the profiling results tree shown above, this last intermediate level corresponds to the keys `fill_kernel-0`, ..., `fill_kernel-5`.
The keys concatenate the kernel name with the index of the corresponding `ncu` [action](https://docs.nvidia.com/nsight-compute/PythonReportInterface/index.html#ncu_report.IAction) representing the kernel launch in the `.ncu-rep` report.
This level exists because an NVTX range may in general contain several kernel launches.

In our example, each innermost NVTX range contains exactly one kernel launch.
Hence, the NVTX range path suffices to identify each kernel launch, and the key of the last intermediate level is not needed.
For such cases, the class {py:class}`~reprospect.tools.ncu.report.ProfilingResults` provides a convenience method {py:meth}`~reprospect.tools.ncu.report.ProfilingResults.query_single_next_metrics` that allows the collected profiling metrics for a kernel launch to be retrieved directly from the NVTX range path, without specifying the key of the last intermediate level.

In [ ]:
key_char_2, metrics_char_2 = results.query_single_next_metrics(accessors=('char_range', 'static_batch_size_2'))

inst_executed_char_2 = metrics_char_2["Executed instructions"]

print(f'Instructions executed for kernel launch {key_char_2}: {inst_executed_char_2}')

Interpreting kernel profiling results
-------------------------------------

From the {py:class}`~reprospect.tools.ncu.report.ProfilingResults` tree, the collected profiling metrics can be readily read into other Python data structures for further post-processing.

Here, we gather the collected performance metrics into a {py:class}`pandas.DataFrame` to help with interpreting the results.

In [ ]:
import pandas as pd

BATCH_SIZES = (1, 2, 4, 8, 16, 32)

metrics_char = {
    batch_size: dict(results.query_single_next_metrics(('char_range', f'static_batch_size_{batch_size}'))[1])
    for batch_size in BATCH_SIZES
}

df_char = pd.DataFrame.from_dict(metrics_char, orient='index')

selected = [
    'Launch grid size x',
    'Executed instructions',
    'Warp stall LG throttle',
    'Device memory store utilization (% of peak elapsed)',
]

print(rich_helpers.to_string(rich_helpers.df_to_table(df_char[selected].T, show_index=True)))

From this representation of the collected profiling metrics, we can readily observe that as the static batch size increases, the number of executed instructions decreases.
Indeed, each warp must execute certain instructions only once, such as those that determine each thread's position in the grid and load the kernel's parameters.
As the static batch size increases, this once-per-warp instruction count is amortized over the batch.

We can bring this observation more to the forefront by calculating the number of executed instructions on a per-lane and per-stored-element basis:

In [ ]:
WARP_SIZE = 32
ELEMENT_COUNT = 1024 * 1024 * 512

print(rich_helpers.to_string(rich_helpers.ds_to_table(df_char['Executed instructions'] * WARP_SIZE / ELEMENT_COUNT)))

The number is highest for the reference kernel.
As the static batch size increases, it decays toward the count needed for the kernel's loop alone, i.e., the instructions that update the address, execute the store, evaluate the loop condition, and branch.

As a result of most of the executed instructions being once-per-warp instructions rather than stores, the reference kernel executes store instructions, and thus feeds the memory system with bytes to be stored, at a rate below the rate that the memory system can sustain.
As the static batch size increases and the once-per-warp instruction count becomes relatively less significant, store instructions are issued more rapidly and the memory system becomes the limiting resource.
This interpretation is also consistent with the increase of the LG throttle warp stall reason metric, which is associated with warps stalled on issuing a local or global memory instruction because the corresponding L1/TEX cache queue is full.

For the type `int`, each store instruction feeds four times the bytes to the memory system, which explains that the reference kernel already sits on the plateau.

In [ ]:
metrics_int = {
    batch_size: dict(results.query_single_next_metrics(('int_range', f'static_batch_size_{batch_size}'))[1])
    for batch_size in BATCH_SIZES
}

df_int = pd.DataFrame.from_dict(metrics_int, orient='index')

print(rich_helpers.to_string(rich_helpers.df_to_table(df_int[selected].T, show_index=True)))

Assertions on kernel profiling results
--------------------------------------

With the collected data gathered in Python data structures, the analysis can go all the way to test assertions.
Here, we encode the main findings of the analysis in test assertions and verify programmatically the performance of the restructured kernel.

First, we verify that the memory access pattern is coalesced at every batch size.
We thus assert that each warp-level global store instruction generates one warp-level global store request that accesses one 32-byte sector (32 lanes {math}`\times` 1 byte) for type `char` and four 32-byte sectors (32 lanes {math}`\times` 4 bytes) for type `int`.

In [ ]:
SECTOR_SIZE = 32 # byte

for (metrics_type, size_of_type) in ((metrics_char, 1), (metrics_int, 4)):
    for batch_size in BATCH_SIZES:
        m = metrics_type[batch_size]
        assert m['L1/TEX cache global store requests'] == m['L1/TEX cache global store sass instructions']
        assert m['L1/TEX cache global store sectors'] / m['L1/TEX cache global store requests'] == WARP_SIZE * size_of_type / SECTOR_SIZE

Next, we verify that the number of executed instructions decreases with the static batch size.

In [ ]:
import itertools

for metrics_type in (metrics_char, metrics_int):
    inst_executed_type = [metrics_type[batch_size]['Executed instructions'] for batch_size in BATCH_SIZES]
    assert all(a >= b for a, b in itertools.pairwise(inst_executed_type))

    # The reduction is significant.
    assert metrics_type[16]['Executed instructions'] < metrics_type[1]['Executed instructions'] / 2

Finally, we verify the performance.
For the static batch sizes belonging to the throughput plateau observed in the unprofiled run, we verify that device-memory write utilization remains above 70% of the peak sustained rate.
The margin of 70% is well below the observed ~85%, to absorb run-to-run noise.

In [ ]:
MIN_DEVICE_MEMORY_WRITE_UTILIZATION_PCT = 70 # percent

for (metrics_type, plateau_batch_sizes_type) in ((metrics_char, (4, 8, 16, 32)), (metrics_int, (1, 2, 4, 8, 16, 32))):
    for batch_size in plateau_batch_sizes_type:
        assert metrics_type[batch_size]['Device memory store utilization (% of peak elapsed)'] >= MIN_DEVICE_MEMORY_WRITE_UTILIZATION_PCT

Going further: insight through binary analysis
----------------------------------------------

Inspecting the compiled CUDA assembly (SASS) code can be helpful to interpret kernel profiling results. 
`ReProspect`'s binary-analysis component can be readily used for this purpose.

Here, we will look at the SASS code to interpret the two warp stall reason metrics that remain to be explained: the short and long scoreboard stalls.

In [ ]:
print(rich_helpers.to_string(rich_helpers.df_to_table(df_char[['Warp stall short scoreboard', 'Warp stall long scoreboard']].T, show_index=True)))

We can observe that the short scoreboard stalls are non-zero for the reference kernel and decrease or remain about the same as the static batch size increases.
The long scoreboard stalls are exactly zero for the reference kernel and increase with the static batch size.

In [ ]:
from reprospect.tools.binaries import CuObjDump
from reprospect.tools.binaries.sass import Decoder

arch = visible_gpus.iloc[0]['architecture']

cuobjdump, _ = CuObjDump.extract(
    file=executable,
    arch=arch,
    cwd=workdir,
    cubin=f'static_batch_size.2.{arch.as_sm}.cubin',
)

for batch_size in (1, 16):
    print(f'Static batch size {batch_size}:\n{Decoder(code=cuobjdump.functions[metrics_char[batch_size]["demangled"]].code)}')

The instructions with opcodes `LDC`, `LDCU`, `S2R` and some of the instructions with opcode `IMAD` are the aforementioned once-per-warp instructions allowing each thread to determine its position in the grid and load the kernel's parameters.
The load and special-register move instructions are variable-latency instructions.
As indicated by the markers in the barrier columns (`b`), they set one of six so-called scoreboard barriers (`Wr`) and dependent instructions wait for their results to be ready (`Wa`).

The instruction with opcode `STG` is the store instruction.
It is also a variable-latency instruction.
It sets a scoreboard barrier (`Re`) if its operands must be protected from being overwritten by subsequent instructions until the instruction has read its operands.

The scoreboard warp stall reason metrics are associated with warps stalled waiting for such scoreboard dependencies.
Nsight Compute [distinguishes](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html?highlight=average#warp-stall-reasons) between long scoreboard stalls associated with waits on L1/TEX cache memory operations (local, global, texture and surface memory) and short scoreboard stalls associated with waits on memory operations other than L1/TEX (thus including the loads from constant memory that we see in our example).

The collected short-scoreboard warp stall reason metric indicates that as the static batch size increases, the number of warp stalls associated with waits on the once-per-warp loads of the kernel parameters decreases or remains about the same.

For the reference kernel, the collected long-scoreboard warp stall reason metric is zero.
Indeed, for this kernel, the store instruction does not set a scoreboard barrier, because no subsequent instruction overwrites its operands.
For the restructured kernel, the collected long-scoreboard warp stall reason metric becomes non-zero and increases with the static batch size.
Indeed, for the restructured kernel, the store instruction sets a scoreboard barrier because it must protect its operands from being overwritten in the next iteration in the batch.
The increase with the static batch size is consistent with the memory system becoming saturated, queued stored instructions taking longer to read their operands, and warps stalling at the instruction updating the address for the next iteration, waiting on the barrier.

Outlook
-------

The kernel restructuring studied in this notebook is modeled on a performance optimization that `Kokkos` recently introduced in its `ViewFill` implementation through the `Kokkos::Experimental::StaticBatchSize` policy.
The [pull request](https://github.com/kokkos/kokkos/pull/8795) featured a benchmark that directly inspired the CUDA source code of this notebook.

As this notebook illustrates, `ReProspect` allows kernel profiling analyses to be concisely scripted, with the focus directed toward the profiling results and their interpretation.
Analyses of this kind can accompany pull requests and complement benchmarks, to motivate code changes and give reviewers insight into the performance impact.
With `ReProspect`'s fully programmatic approach, such analyses can be incorporated as tests in CI/CD pipelines that can detect the analysis ceasing to hold or performance regressions as compiler toolchains, libraries, and hardware evolve.